In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

# directory that contains agent_tools.py (must run before importing agent_tools)
_pkg = Path.cwd()
# print(_pkg)
if (_pkg / "agent_tools.py").is_file():
    pkg = _pkg
else:
    pkg = _pkg.parent  # typical: cwd is .../notebook, module is one level up

sys.path.insert(0, str(pkg.resolve()))

# Drop stale cache so edits to agent_tools.py are picked up without restarting the kernel
sys.modules.pop("agent_tools", None)
import agent_tools as at
# from agent_tools import AgentMemory


# Input Data
Loading dataframe from a parquet or csv data located in path

In [ ]:
path_data = '..\data\heloc_dataset_v1.parquet'

In [ ]:
try:
    df = pd.read_parquet(path_data)
except:
    try:
        df = pd.read_csv(path_data)
    except:
        raise ValueError(f"Failed to load data from {path_data}")

# Metadata
Setting metadata for column aliases, target column, feature column, time related column, data type (for data split), etc. The purpose of this aliases is so that we don't need to manually change the column name one-by-obne if we want to use different column for label, features, etc.  

col_time: datetime of event. e.g, date of application, click date, etc.  
col_target: column name of groundtruth label.  
cols_feat: column name of features used for training.  
col_day: column name of day-truncated col_time.  
col_month: column name of month-truncated col_time.  
col_type: column name of data type, whether the data is for train, test, valid, or hold-out sample (hoot, oot).  
col_score: column name of predicted probability score.
cols_feat_num: column name of numerical features.  
cols_feat_cat: column name of categorical features.  
cols_feat_time: column name of datetime features.


In [ ]:
col_time = 'date'
col_target = 'label'
cols_feat = ['ExternalRiskEstimate', 'MSinceOldestTradeOpen',
       'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades',
       'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec',
       'PercentTradesNeverDelq', 'MSinceMostRecentDelq',
       'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades',
       'NumTradesOpeninLast12M', 'PercentInstallTrades',
       'MSinceMostRecentInqexcl7days', 'NumInqLast6M', 'NumInqLast6Mexcl7days',
       'NetFractionRevolvingBurden', 'NetFractionInstallBurden',
       'NumRevolvingTradesWBalance', 'NumInstallTradesWBalance',
       'NumBank2NatlTradesWHighUtilization', 'PercentTradesWBalance']
col_day = 'day'
col_month = 'month'
col_type = 'type'
col_score = 'score'

In [ ]:
df[col_day] = at.get_day_from_date(df, col_time)
df[col_month] = at.get_month_from_date(df, col_time)
cols_feat_num, cols_feat_cat, cols_feat_time = at.get_feature_dtype(df, cols_feat)


# EDA
Explarotary Data Analysis: Understanding the data quality, feature distribution, and labels before start modelling.  
In this session we also do preliminary filtering on the data by making sure that:  
1. Features has enough penetration: Missing rate less than 80%

## Missing rate of features

In [ ]:
df_nan_rate = at.get_nan_rate(df, cols_feat)
display(df_nan_rate)

## Missing rate of features over period of time.

In [ ]:
df_nan_rate_timely = at.get_nan_rate_timely(df, cols_feat, col_month)
display(df_nan_rate_timely)

# Split Data

In this section we split data into several data type:  

oot: Out of time sample, to check the performance of the model after development data.  
hoot: Historical out of time sample, to check the performance of the model before the development data.    
train: data sample for training model.  
valid: data sample for hyperparameter tuning.  
test: data sample for on-paper evaluation. 
In Summary, data type must follow thiss equence of event: oot --> train (t-3) | test | valid (t-2)  --> oot (t-1), where t is time notation.  

By looking at:
1. Distribution of target rate over period of time.  
2. Data count over the period of time.  
3. Number of positive label over the period of time.  

In the case of limited data, we should prioritize the data split with the following order:  
1. train (for model building: should have minimum 1000 data count) and oot data (for performance evaluation: should have min 300 data count).  
2. test.  
3. valid.  
4. hoot.  

### Target (groundtruth) rate over period of time

In [ ]:
df_stats_target_mean_per_month = at.get_timely_binary_target_rate(df, col_month, col_target)
display(df_stats_target_mean_per_month)

In [ ]:
oot_th = 201510
hoot_th = 201503

In [ ]:
# Use the same integer scale for `col_period` and for oot_th / hoot_th (here: `month`).
df[col_type] = at.split_data(df, col_target, col_month, oot_th, hoot_th)

In [ ]:
df[col_type].value_counts()

Checking the consistency of the label distribution in each sample type.

In [ ]:
df_target_rate_per_sample = at.get_target_rate_sample(df, col_type, col_target)
display(df_target_rate_per_sample)


# Feature transformation
1. Transform the feature into Weight of Evidence to handle non-monotinicity and inbalanced label.  
2. Fine tune the WoE if necessary based on: number of data each bin, number of positive label each bin, number of positive rate (target mean) each bin.  

In [ ]:
# Getting binningprocess object from otpbinning for woe transformation, woe statistics from training sample, and list of features that have issues during data transformation (missing feature, etc)
dict_bin, bp, df_binning_stats, binning_feature_issues = at.get_optimal_bin(
    df.loc[df[col_type] == "train"], cols_feat , col_target, cols_feat_cat=cols_feat_cat
)
display(df_binning_stats.sort_values(by='gini_power', ascending=False))

In [ ]:
# Getting transformed woe data, name of columns containing woe data, and list of features that have issues during data transformation (missing feature, etc)
# X_woe, cols_feat_woe, features_issue_woe = at.get_woe_from_bp(df, cols_feat, bp)

In [ ]:
# Getting binning table containing binning statistics for every feature. This statistics is used for fine-tuning if necessary.
df_binning_tables, features_issue_binning_tables = at.get_binning_tables_from_bp(bp, cols_feat)
display(df_binning_tables)

In [ ]:
dict_bin

In [ ]:
# Fine-tune WOE if necessary. If not, skip it.
dict_bin = {'ExternalRiskEstimate': [64.5, 68.5, 70.5, 74.5, 80.5],
 'MSinceOldestTradeOpen': [31.5, 107.5, 134.5, 212.5, 273.5],
 'MSinceMostRecentTradeOpen': [2.5, 3.5, 7.5, 9.5, 16.5],
 'AverageMInFile': [52.5, 65.5, 80.5, 97.5, 137.5],
 'NumSatisfactoryTrades': [5.5, 9.5, 11.5, 19.5, 39.5],
 'NumTrades60Ever2DerogPubRec': [0.5, 1.5, 2.5],
 'NumTrades90Ever2DerogPubRec': [0.5, 1.5],
 'PercentTradesNeverDelq': [63.5, 75.5, 83.5, 90.5, 95.5],
 'MSinceMostRecentDelq': [-3.5, 2.5, 15.5, 22.5, 46.5],
 'MaxDelq2PublicRecLast12M': [1.5, 5.5, 6.5],
 'MaxDelqEver': [3.5, 5.5, 6.5],
 'NumTotalTrades': [8.5, 11.5, 21.5, 24.5, 45.5],
 'NumTradesOpeninLast12M': [0.5, 1.5, 2.5, 3.5, 5.5],
 'PercentInstallTrades': [24.5, 35.5, 42.5, 46.5, 65.5],
 'MSinceMostRecentInqexcl7days': [-7.5, 0.5, 1.5, 7.5, 12.5],
 'NumInqLast6M': [0.5, 1.5, 2.5, 4.5],
 'NumInqLast6Mexcl7days': [0.5, 1.5, 2.5, 4.5],
 'NetFractionRevolvingBurden': [14.5, 26.5, 39.5, 65.5, 75.5],
 'NetFractionInstallBurden': [7.5, 33.5, 61.5, 72.5, 93.5],
 'NumRevolvingTradesWBalance': [1.5, 2.5, 4.5, 6.5, 7.5],
 'NumInstallTradesWBalance': [-3.5, 1.5, 2.5, 3.5, 4.5],
 'NumBank2NatlTradesWHighUtilization': [0.5, 1.5, 2.5, 3.5],
 'PercentTradesWBalance': [44.5, 64.5, 75.5, 84.5, 89.5]}

bp, df_binning_stats, binning_feature_issues = at.modify_optimal_bin(df, dict_bin, cols_feat, col_target, cols_feat_cat)

In [ ]:
# Transform cols_feat to woe value and store it as new columns in the existing dataframe 
df[cols_feat_woe] = bp.transform(df[cols_feat], metric="woe")
display(df[cols_feat_woe])

# Feature Selection
In this section we implement different methodology for filtering the input feature so that only feature that has:
1. stability over time 
2. good performance  
can enter the model.

In [ ]:
# Filtering for stability. We should as possible as we can minimize the target rate cross-over in feature bin over period of the time.
df_stats = at.get_timely_target_rate_feature_segment(df.loc[df[col_type]=='train'], cols_feat_woe, col_target, col_month)
display(df_stats)

In [ ]:
# Pick only the desired features.
cols_feat_woe = at.set_value_list(['ExternalRiskEstimate_woe',
 'MSinceOldestTradeOpen_woe',
 'MSinceMostRecentTradeOpen_woe',
 'AverageMInFile_woe',
 'NumSatisfactoryTrades_woe',
 'NumTrades60Ever2DerogPubRec_woe',
 'NumTrades90Ever2DerogPubRec_woe',
 'PercentTradesNeverDelq_woe',
 'MSinceMostRecentDelq_woe',
 'MaxDelq2PublicRecLast12M_woe',
 'MaxDelqEver_woe',
 'NumTotalTrades_woe',
 'NumTradesOpeninLast12M_woe',
 'PercentInstallTrades_woe',
 'MSinceMostRecentInqexcl7days_woe',
 'NumInqLast6M_woe',
 'NumInqLast6Mexcl7days_woe',
 'NetFractionRevolvingBurden_woe',
 'NetFractionInstallBurden_woe',
 'NumRevolvingTradesWBalance_woe',
 'NumInstallTradesWBalance_woe',
 'NumBank2NatlTradesWHighUtilization_woe',
 'PercentTradesWBalance_woe'])

In [ ]:

selected_feats_auc, feats_stats_auc = at.select_features_auc_max_corr(df.loc[df[col_type]=='train'], cols_feat_woe, col_target, max_corr=0.5)
display(feats_stats_auc)

In [ ]:
selected_feats_iv, feats_stats_iv = at.select_features_iv_max_corr(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(feats_stats_iv)

In [ ]:
selected_feats_aic_fwd, df_stats_aic_fwd = at.select_features_aic_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(df_stats_aic_fwd)

In [ ]:
selected_feats_aic_bwd, df_stats_aic_bwd = at.select_features_aic_backward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(df_stats_aic_bwd)

In [ ]:
selected_feats_bic_bwd, df_stats_bic_bwd = at.select_features_bic_backward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(df_stats_bic_bwd)

In [ ]:
selected_feats_bic_fwd, df_stats_bic_fwd = at.select_features_bic_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(selected_feats_bic_fwd)

In [ ]:
selected_feats_auc_fwd, df_stats_auc_fwd = at.select_features_auc_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)
display(selected_feats_auc_fwd)

# Model training

In [ ]:
dict_statistics_auc, best_model_auc, importance_auc = at.train_logreg_l2_tune_cv(df, selected_feats_auc, col_target, col_type)
dict_statistics_iv, best_model_iv, importance_iv = at.train_logreg_l2_tune_cv(df, selected_feats_iv, col_target, col_type)
dict_statistics_aic_fwd, best_model_aic_fwd, importance_aic_fwd = at.train_logreg_l2_tune_cv(df, selected_feats_aic_fwd, col_target, col_type)
dict_statistics_aic_bwd, best_model_aic_bwd, importance_aic_bwd = at.train_logreg_l2_tune_cv(df, selected_feats_aic_bwd, col_target, col_type)
dict_statistics_bic_fwd, best_model_bic_fwd, importance_bic_fwd = at.train_logreg_l2_tune_cv(df, selected_feats_bic_fwd, col_target, col_type)
dict_statistics_bic_bwd, best_model_bic_bwd, importance_bic_bwd = at.train_logreg_l2_tune_cv(df, selected_feats_bic_bwd, col_target, col_type)
dict_statistics_auc_fwd, best_model_auc_fwd, importance_auc_fwd = at.train_logreg_l2_tune_cv(df, selected_feats_auc_fwd, col_target, col_type)

In [ ]:
best_model_iv.best_params_

# Model Prediction

In [ ]:
df['proba_0_auc'], df['proba_1_auc'], feature_predict_issue = at.logreg_predict(df, best_model_auc, cols_feat_woe)
df['proba_0_iv'], df['proba_1_iv'], feature_predict_issue = at.logreg_predict(df, best_model_iv, cols_feat_woe)
df['proba_0_aic_fwd'], df['proba_1_aic_fwd'], feature_predict_issue = at.logreg_predict(df, best_model_aic_fwd, cols_feat_woe)
df['proba_0_aic_bwd'], df['proba_1_aic_bwd'], feature_predict_issue = at.logreg_predict(df, best_model_bic_fwd, cols_feat_woe)
df['proba_0_bic_fwd'], df['proba_1_bic_fwd'], feature_predict_issue = at.logreg_predict(df, best_model_aic_bwd, cols_feat_woe)
df['proba_0_bic_bwd'], df['proba_1_bic_bwd'], feature_predict_issue = at.logreg_predict(df, best_model_bic_bwd, cols_feat_woe)
df['proba_0_auc_fwd'], df['proba_1_auc_fwd'], feature_predict_issue = at.logreg_predict(df, best_model_auc_fwd, cols_feat_woe)

# Model Evaluation

In [ ]:
cols_score = ['proba_1_auc', 'proba_1_iv', 'proba_1_aic_fwd', 'proba_1_aic_bwd', 'proba_1_bic_fwd', 'proba_1_bic_bwd', 'proba_1_auc_fwd']
df_stats_performance_per_scores = at.compare_scores_gini_auc(df, col_type, cols_score, col_target)
display(df_stats_performance_per_scores.transpose())

In [ ]:
df_stats_score_psi_over_time = at.get_timely_vars_psi(df.loc[df[col_type]=='train'], df.loc[df[col_type]!='train'], cols_score, col_month, n_bins=5)
display(df_stats_score_psi_over_time)

# Deep dive individual score
After we chose the score we want to use, we can do further analysis as the final check.

In [ ]:
df_stats = at.get_score_predictive_power_data_type(df, 'proba_1_iv', col_type, col_target)
display(df_stats)

In [ ]:
df_stats = at.get_score_predictive_power_timely(df, 'proba_1_iv', col_month, col_target)
display(df_stats)

In [ ]:
df_stats = at.get_score_predictive_power_data_type_bootstrap(df, 'proba_1_iv', col_type, col_target)

In [ ]:
df_stats

In [ ]:
df_stats = at.compare_score_predictive_power_data_type_bootstrap(df, 'proba_1_iv', 'proba_1_auc', col_type, col_target)

In [ ]:
df_stats

In [ ]:
df_stats = at.get_timely_feature_psi_woe(df.loc[df[col_type]=='train'], df.loc[df[col_type]!='train'], cols_feat_woe, col_month)

In [ ]:
df_stats

In [ ]:
df_stats = at.get_timely_psi(df.loc[df[col_type]=='train'], df.loc[df[col_type]!='train'], 'proba_1', col_month)

In [ ]:
df_stats